In [1]:
import geopandas
from shapely.geometry import LineString
import pandas as pd
import os

In [2]:
def match(geom1, geom2, threshold1, threshold2):
    if geom1.intersects(geom2):
        g1 = geom1.intersection(geom2)
        area = g1.area
        return area > threshold1/2.0 and area > threshold2 * min(geom1.area, geom2.area)
    return False

def match_tables(table1, table2, threshold):
    min_area = min(table1['geometry'].apply(lambda x: x.area).min(),table2['geometry'].apply(lambda x: x.area).min())
    newdata = pd.DataFrame(columns = ['id', 'id1', 'id2', 'geometry'])
    count = 0
    for index1, row1 in table1.iterrows():
        g1 = row1['geometry']
        if count == 1:
            break
        for index2, row2 in table2.iterrows():
            g2 = row2['geometry']
            if match(g1, g2, min_area, threshold):
                line = LineString([g1.centroid,g2.centroid])
                newmatch = {'id':len(newdata), 'id1':index1, 'id2':index2, 'geometry':line}
                newdata.loc[len(newdata)] = newmatch
                if newmatch['id'] == 20:
                    count = 1
                    break
    return geopandas.GeoDataFrame(newdata, geometry='geometry')

In [3]:
p1 = geopandas.read_file('../data/matching/ilots_verniquet.shp')
p2 = geopandas.read_file('../data/matching//ilots_vasserot.shp')

In [4]:
newdata = match_tables(p1,p2,0.2)
newdata.set_crs(p1.crs, inplace=True)

Final count:  21


,id,id1,id2,geometry
0,0,0,1161,"LINESTRING (650243.978 6864670.444, 650306.320..."
1,1,1,1161,"LINESTRING (650425.108 6864926.845, 650306.320..."
2,2,2,1161,"LINESTRING (650255.298 6864894.974, 650306.320..."
3,3,3,1161,"LINESTRING (650050.737 6864819.481, 650306.320..."
4,4,4,1162,"LINESTRING (649378.949 6864558.304, 649385.417..."
5,5,5,1237,"LINESTRING (649915.332 6864507.791, 649789.913..."
6,6,6,1158,"LINESTRING (650606.765 6864489.048, 650629.443..."
7,7,6,1159,"LINESTRING (650606.765 6864489.048, 650666.102..."
8,8,6,1161,"LINESTRING (650606.765 6864489.048, 650306.320..."
9,9,7,1057,"LINESTRING (650971.151 6865072.607, 650880.017..."


In [5]:
newdata.explore()